# Lab 03 — 分離技術與運氣：行為變異性

**課程**：player-behavior-analytics（玩家行為分析）／第 3 課 分離技術與運氣：縱向分析框架
**目的**：用「下注行為的變異性」判斷一位玩家的下注是出於可重現的規則（技術訊號）還是隨機擺動（運氣主導）。
**方式**：全程以 Gemini（AI 助手）產生程式碼——把每個任務的自然語言描述貼給 Gemini，再把生成的程式碼貼進下方 code cell 執行；**重點是觀察結果**，不是寫程式。每個任務下方附參考程式碼，可先自行生成再比對。

> 執行：在 Google Colab 上傳本 .ipynb（或直接開啟），Runtime → Run all。本範本內建模擬數據產生器，無需上傳檔案。

## 0. 載入數據

執行下方 cell 產生模擬數據。本 Lab 保留每位玩家的真實原型欄位（archetype），在最後一步做「方法 vs 真相」對照。

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    """模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    """
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df

df = gen_baccarat(seed=42)
df.head()


## 任務 1 — 計算行為變異性（CV）

核心想法：有固定策略的玩家，下注金額穩定（變異 ≈ 0）；依情境調整的玩家，變異中等；隨機下注的玩家，變異最大。

在 Gemini 輸入：

> 「按玩家分組，計算每位玩家下注金額的變異係數（CV = 標準差 / 平均），並排序顯示。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
cv = (df.groupby("player_id")["bet_amount"]
        .agg(lambda s: s.std() / s.mean())
        .reset_index(name="cv_bet")
        .sort_values("cv_bet"))
cv.round(3)

**觀察**：CV 自動分成三層——有人 0.00（每局同一金額），有人約 0.4（依情境調整），有人 0.5 以上（金額大幅擺動）。

## 任務 2 — 分層：低／中／高變異

在 Gemini 輸入：

> 「把玩家按 CV 分成三類：低變異（<0.1）、中變異（0.1–0.44）、高變異（>0.44），並列出每類玩家。」

> 閾值依資料分布而定——本範例數據的天然分層落在 0.00／0.40–0.43／0.45–0.60。

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
cv["type"] = pd.cut(cv["cv_bet"], bins=[0, 0.1, 0.44, np.inf],
                    labels=["低變異（固定規則）", "中變異（條件規則）", "高變異（隨機擺動）"],
                    include_lowest=True)
cv.round(3)

## 任務 3 — 視覺對照：滾動平均折線圖

在 Gemini 輸入：

> 「畫 3 位不同變異層級玩家的逐局下注 10 局滾動平均折線圖（同圖比較）。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
import matplotlib.pyplot as plt

for pid in ["P00", "P02", "P04"]:   # 分別為低／中／高變異代表
    sub = df[df["player_id"] == pid].sort_values("timestamp").reset_index(drop=True)
    plt.plot(sub["bet_amount"].rolling(10).mean(), label=pid)
plt.legend()
plt.title("Rolling mean of bet amount (window=10)")
plt.xlabel("round"); plt.ylabel("bet amount")
plt.show()

**觀察**：低變異玩家的滾動平均是一條水平線；高變異玩家的滾動平均起伏劇烈——即使把 10 局平均，隨機性依然可見。

## 任務 4 — 方法 vs 真相：與真實原型對照

在 Gemini 輸入：

> 「把 CV 分層結果與每位玩家的原型（archetype 欄位）交叉比對，統計每層有哪些原型。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
truth = df[["player_id", "archetype"]].drop_duplicates()
cv.merge(truth, on="player_id").groupby(["type", "archetype"]).size().unstack(fill_value=0)

**討論**：
- 低變異 = 紀律型＋謬誤型（金額固定——但謬誤型的「問題」在於押注方向，金額看不出來）
- 高變異 = 追注型＋波動型——兩者金額變異同樣大，但成因完全不同（追注是規則性反應，波動是隨機）——單看 CV 無法分離
- 結論：變異性是「技術訊號」的第一層篩子；要分辨「規則性加注」與「隨機擺動」，需要看序列結構（第 4 課）

**進階觀察（縱向粒度）**：把變異性改算在「每 session 平均下注」層級——跨 session 的變異比局級變異小得多，說明差異主要來自「局內節奏」，不是「跨 session 改變」。

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
# 進階：跨 session 層級的變異（每 session 平均下注的 CV）
(df.groupby(["player_id", "session_id"])["bet_amount"].mean()
   .groupby("player_id").agg(lambda s: s.std() / s.mean()))